# Aplicação — Classificação de chamados de help desk
**Gustavo Carvalho de Oliveira | UDF | Profa. Kadidja Valéria | 2026.2**

Pergunta: é possível sugerir a **fila de atendimento** de um chamado de TI a partir do texto que o usuário escreve na abertura? Todos os chamados são **sintéticos**, gerados abaixo.

In [1]:
import sys, json, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt, sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
SEMENTE = 42
print('Python', sys.version.split()[0], '| scikit-learn', sklearn.__version__, '| pandas', pd.__version__, '| NumPy', np.__version__)

Python 3.12.3 | scikit-learn 1.9.1 | pandas 3.0.6 | NumPy 2.5.4


## 1. Geração dos chamados sintéticos
Filas com tamanhos diferentes (desbalanceamento), 12% de textos vagos e 8% de filas trocadas (erro de triagem).

In [2]:
rng = np.random.default_rng(SEMENTE)
vocab = {
 'Acesso e senha': (170, ['senha expirada', 'nao consigo logar', 'conta bloqueada', 'resetar senha', 'acesso negado no sistema',
                          'esqueci minha senha', 'liberar acesso a pasta', 'usuario bloqueado', 'autenticacao falhou', 'token mfa nao chega']),
 'Rede e internet': (150, ['internet caiu', 'wifi sem conexao', 'rede lenta', 'vpn nao conecta', 'sem acesso a internet',
                          'cabo de rede', 'wi-fi do laboratorio', 'ping alto', 'site nao abre', 'conexao instavel']),
 'Impressora':      (90,  ['impressora nao imprime', 'papel atolado', 'toner acabando', 'fila de impressao travada', 'impressora offline',
                          'scanner nao digitaliza', 'imprimir frente e verso', 'driver da impressora', 'impressao borrada', 'multifuncional parada']),
 'Hardware':        (110, ['computador nao liga', 'monitor sem imagem', 'teclado com defeito', 'mouse parou', 'notebook esquentando',
                          'barulho no gabinete', 'tela azul', 'bateria nao carrega', 'hd cheio', 'memoria ram']),
 'Software':        (140, ['instalar programa', 'licenca do office', 'excel travando', 'atualizar windows', 'erro ao abrir aplicativo',
                          'instalar python', 'antivirus bloqueou', 'teams nao abre', 'programa fecha sozinho', 'versao do navegador']),
 'E-mail':          (100, ['email nao chega', 'caixa de entrada cheia', 'outlook nao sincroniza', 'spam na caixa', 'anexo nao envia',
                          'configurar email no celular', 'lista de distribuicao', 'email retornou', 'assinatura de email', 'phishing suspeito'])}
extras = ['urgente', 'por favor', 'desde ontem', 'sala 204', 'bloco b', 'no laboratorio', 'de novo', 'preciso hoje', '', '', '']
vagos = ['nao funciona', 'problema', 'ajuda', 'erro', 'parou de funcionar', 'socorro ti']
filas = list(vocab); linhas = []
for fila, (qtd, frases) in vocab.items():
    for _ in range(qtd):
        txt = rng.choice(frases)
        if rng.random() < 0.5: txt = rng.choice(frases) + ' e ' + txt if rng.random() < 0.15 else txt + ' ' + rng.choice(extras)
        if rng.random() < 0.12: txt = rng.choice(vagos) + (' ' + rng.choice(extras) if rng.random() < 0.5 else '')
        rotulo = fila if rng.random() >= 0.08 else rng.choice([f for f in filas if f != fila])
        linhas.append({'texto': txt.strip(), 'fila': rotulo})
df = pd.DataFrame(linhas).sample(frac=1, random_state=SEMENTE).reset_index(drop=True)
df.to_csv('../dados/helpdesk_chamados_sinteticos.csv', index=False)
print(df.shape); print(df['fila'].value_counts().to_string()); print(df.head(6).to_string(index=False))

(760, 2)
fila
Acesso e senha     174
Rede e internet    153
Software           135
Hardware           108
E-mail             101
Impressora          89
                           texto           fila
imprimir frente e verso sala 204     Impressora
          scanner nao digitaliza         E-mail
               nao consigo logar Acesso e senha
               tela azul urgente       Hardware
          liberar acesso a pasta Acesso e senha
                     memoria ram       Hardware


## 2. Divisão estratificada 75/25

In [3]:
X_tr, X_te, y_tr, y_te = train_test_split(df['texto'], df['fila'], test_size=0.25, stratify=df['fila'], random_state=SEMENTE)
print('Treino', len(X_tr), '| Teste', len(X_te))

Treino 570 | Teste 190


## 3. Seleção do modelo só no treino (F1 macro, validação cruzada 5 partes)
Vetorização TF-IDF de palavras + n-gramas de caracteres **dentro do Pipeline**.

In [4]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEMENTE)
def vet():
    return FeatureUnion([('pal', TfidfVectorizer(ngram_range=(1, 2))),
                         ('car', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4)))])
cands = {'Referência (mais frequente)': DummyClassifier(strategy='most_frequent')}
for a in [0.1, 0.5, 1.0]: cands[f'Naive Bayes alpha={a}'] = Pipeline([('tfidf', vet()), ('clf', MultinomialNB(alpha=a))])
for c in [1, 10]: cands[f'Regressão logística C={c}'] = Pipeline([('tfidf', vet()), ('clf', LogisticRegression(C=c, max_iter=2000))])
res = []
for nome, m in cands.items():
    s = cross_validate(m, X_tr, y_tr, cv=cv, scoring=['accuracy', 'f1_macro'], return_train_score=True)
    res.append({'modelo': nome, 'acc_treino': s['train_accuracy'].mean(), 'acc_val': s['test_accuracy'].mean(),
                'desvio': s['test_accuracy'].std(), 'f1_val': s['test_f1_macro'].mean()})
res = pd.DataFrame(res); print(res.round(4).to_string(index=False))
nome_esc = res.iloc[1:].sort_values('f1_val', ascending=False).iloc[0]['modelo']; print('Escolhido:', nome_esc)
modelo = cands[nome_esc]

                     modelo  acc_treino  acc_val  desvio  f1_val
Referência (mais frequente)      0.2281   0.2281  0.0000  0.0619
      Naive Bayes alpha=0.1      0.8693   0.8281  0.0226  0.8221
      Naive Bayes alpha=0.5      0.8702   0.8316  0.0231  0.8269
      Naive Bayes alpha=1.0      0.8697   0.8316  0.0231  0.8272
    Regressão logística C=1      0.8798   0.8316  0.0231  0.8276
   Regressão logística C=10      0.9053   0.8070  0.0229  0.7997
Escolhido: Regressão logística C=1


## 4. Limiar de confiança escolhido no treino
Probabilidades fora da dobra (cross_val_predict) para decidir, **sem tocar no teste**, a partir de que confiança o sistema encaminha automaticamente.

In [5]:
proba_cv = cross_val_predict(modelo, X_tr, y_tr, cv=cv, method='predict_proba')
classes = np.array(sorted(y_tr.unique())); pred_cv = classes[proba_cv.argmax(1)]; conf_cv = proba_cv.max(1)
for lim in [0.4, 0.5, 0.6, 0.7]:
    m = conf_cv >= lim
    print(f'limiar {lim}: cobertura {m.mean():.3f} | acerto nos encaminhados {(pred_cv[m] == y_tr.values[m]).mean():.3f}')

limiar 0.4: cobertura 0.891 | acerto nos encaminhados 0.894
limiar 0.5: cobertura 0.805 | acerto nos encaminhados 0.932
limiar 0.6: cobertura 0.726 | acerto nos encaminhados 0.935
limiar 0.7: cobertura 0.572 | acerto nos encaminhados 0.929


## 5. Teste final (uma única vez)

In [6]:
LIMIAR = 0.6
modelo.fit(X_tr, y_tr); prev = modelo.predict(X_te)
ref = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
print(f'Acurácia teste {accuracy_score(y_te, prev):.4f} | F1 macro {f1_score(y_te, prev, average="macro"):.4f}')
print(f'Referência: acurácia {accuracy_score(y_te, ref.predict(X_te)):.4f} | F1 macro {f1_score(y_te, ref.predict(X_te), average="macro"):.4f}')
print(classification_report(y_te, prev, digits=3))
conf = modelo.predict_proba(X_te).max(1); m = conf >= LIMIAR
print(f'Limiar {LIMIAR}: cobertura {m.mean():.3f} | acerto nos encaminhados {(prev[m] == y_te.values[m]).mean():.3f}')
cm = confusion_matrix(y_te, prev, labels=classes)
fig, ax = plt.subplots(figsize=(8, 7)); ConfusionMatrixDisplay(cm, display_labels=classes).plot(ax=ax, cmap='Greens', xticks_rotation=45, colorbar=False)
plt.title('Help desk — teste final (sintético)'); plt.tight_layout(); plt.savefig('../assets/helpdesk-matriz.png', dpi=120); plt.close()
ex = ['senha do sistema expirou', 'impressora da sala 204 nao imprime', 'outlook travando', 'nao funciona', 'wifi caiu e nao consigo logar']
for t in ex:
    p = modelo.predict_proba([t])[0]; print(f'{t!r} -> {classes[p.argmax()]} ({p.max():.2f})')
json.dump({'modelo': nome_esc, 'acc_teste': float(accuracy_score(y_te, prev)), 'f1_macro_teste': float(f1_score(y_te, prev, average='macro'))},
          open('resultados_helpdesk.json', 'w'), indent=2, ensure_ascii=False)

Acurácia teste 0.8421 | F1 macro 0.8443
Referência: acurácia 0.2316 | F1 macro 0.0627
                 precision    recall  f1-score   support

 Acesso e senha      0.841     0.841     0.841        44
         E-mail      0.750     0.840     0.792        25
       Hardware      0.889     0.889     0.889        27
     Impressora      0.947     0.818     0.878        22
Rede e internet      0.914     0.842     0.877        38
       Software      0.757     0.824     0.789        34

       accuracy                          0.842       190
      macro avg      0.850     0.842     0.844       190
   weighted avg      0.848     0.842     0.843       190

Limiar 0.6: cobertura 0.811 | acerto nos encaminhados 0.916


'senha do sistema expirou' -> Acesso e senha (0.75)
'impressora da sala 204 nao imprime' -> Impressora (0.80)
'outlook travando' -> Software (0.38)
'nao funciona' -> Acesso e senha (0.50)
'wifi caiu e nao consigo logar' -> Acesso e senha (0.64)
